# Réseau de neurones simple avec TensorFlow

Dans ce TP, nous allons construire deux réseaux de neurones avec l'API de base de TensorFlow (`tf.Variable`, `tf.Module`, `tf.GradientTape`, `tf.data`), sans modèle Keras : un réseau sans couche cachée d'abord, puis un perceptron multicouche (PMC). Le TP Keras a construit les mêmes réseaux avec `compile` et `fit` : celui-ci écrit à la main ce que fait `fit`.

Les données sont celles de MNIST, que le TP Keras a explorées : des images de 28 $\times$ 28 pixels en niveaux de gris, qui représentent des chiffres de 0 à 9. Chaque image est associée à un label, le chiffre que l'image est censée représenter.

Les étapes :

1. Prendre en main les tenseurs TensorFlow
2. Retrouver les coefficients d'une droite par descente de gradient, avec `tf.GradientTape`
3. Charger et préparer les données, mettre de côté un ensemble de validation
4. Écrire un réseau sans couche cachée, sa perte, son exactitude et son pipeline `tf.data`
5. L'entraîner par descente de gradient écrite à la main, puis avec un optimiseur Keras
6. Compiler le pas d'entraînement avec `tf.function`, et compter ses traçages
7. Passer à un PMC et le régulariser
8. Suivre ses entraînements dans TensorBoard, choisir son learning rate sur la validation
9. L'évaluer sur l'ensemble de test, le sauvegarder et le restaurer

Ce notebook n'a pas besoin de GPU : chaque entraînement dure de quelques secondes à quelques minutes sur le processeur de Colab.

## Imports

In [ ]:
import time
from collections.abc import Callable

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Mêmes tirages d'une exécution à l'autre : poids initiaux, mélanges
keras.utils.set_random_seed(0)

TensorFlow fournit les tenseurs, les gradients et le pipeline de données ; Keras, son jeu MNIST, sa fonction de perte, ses optimiseurs, et la graine des tirages aléatoires (`keras.utils.set_random_seed`), pour que vos chiffres soient ceux des solutions. [`matplotlib`](https://matplotlib.org/) et [`numpy`](https://numpy.org/) servent aux graphiques et aux opérations simples sur les tableaux.

## Prise en main des tenseurs

Sur Colab, TensorFlow est déjà installé. Sur votre machine, installez-le avec `pip install tensorflow`.

Avant de construire des réseaux, manipulons les tenseurs TensorFlow, dont l'API est proche de celle de NumPy. Les opérations qui échouent lèvent une `tf.errors.InvalidArgumentError` : attrapez-la avec `try`/`except` pour afficher son message.

1. Créez un tenseur `a` de forme (2, 3) contenant les entiers de 0 à 5 (`tf.range`, puis `tf.reshape`). Quel est son `dtype` ? Convertissez-le en `float32` avec `tf.cast`, puis créez un tenseur `b` de forme (3,) rempli de 1 (`tf.ones`).
2. Calculez `a + b` et le produit matriciel `a @ tf.ones((3, 2))`. Quelles sont les formes des résultats ? Pourquoi `a + tf.ones(2)` échoue-t-il ? Et `a @ b` ?
3. Extrayez la deuxième colonne de `a`, puis sa dernière ligne. Ajoutez une dimension de taille 1 en tête de `a` avec `tf.expand_dims` : quelle forme obtient-on ? À quoi sert une telle dimension avec un modèle ?
4. Convertissez `a` en tableau NumPy (`.numpy()`), puis de nouveau en tenseur (`tf.convert_to_tensor`). Convertissez de même `np.ones(3)` : quel `dtype` obtient-on, et que se passe-t-il si on l'ajoute à `a` ? Enfin, listez les GPU que voit TensorFlow (`tf.config.list_physical_devices`).

**Documentation utile**

- [`tf.range`](https://www.tensorflow.org/api_docs/python/tf/range)
- [`tf.reshape`](https://www.tensorflow.org/api_docs/python/tf/reshape)
- [`tf.ones`](https://www.tensorflow.org/api_docs/python/tf/ones)
- [`tf.cast`](https://www.tensorflow.org/api_docs/python/tf/cast)
- [`tf.convert_to_tensor`](https://www.tensorflow.org/api_docs/python/tf/convert_to_tensor)
- [`tf.expand_dims`](https://www.tensorflow.org/api_docs/python/tf/expand_dims)
- [`tf.config.list_physical_devices`](https://www.tensorflow.org/api_docs/python/tf/config/list_physical_devices)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tf.reshape(tf.range(6), (2, 3))`, puis `tf.cast(a, tf.float32)`
- `a[:, 1]` : la deuxième colonne ; `a[-1]` : la dernière ligne
- `tf.expand_dims(a, axis=0)` : une dimension de taille 1 en position 0
- `tf.config.list_physical_devices("GPU")` : la liste des GPU visibles, vide sans GPU
- Une opération qui échoue, dans un `try` :

```python
try:
  a + tf.ones(2)
except tf.errors.InvalidArgumentError as erreur:
  print(erreur)
```

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
a = tf.reshape(..., (2, 3))
print(a.dtype)
a = tf.cast(a, ...)
b = ...

print((a + b).shape, (a @ tf.ones((3, 2))).shape)
try:
  ...  # a + tf.ones(2), puis a @ b dans un second try
except tf.errors.InvalidArgumentError as erreur:
  print(erreur)

print(a[:, 1], ...)  # deuxième colonne, dernière ligne
print(tf.expand_dims(a, axis=0).shape)

print(tf.convert_to_tensor(a.numpy()).dtype, tf.convert_to_tensor(np.ones(3)).dtype)
try:
  ...  # a + le tenseur créé depuis np.ones(3)
except tf.errors.InvalidArgumentError as erreur:
  print(erreur)
print(tf.config.list_physical_devices(...))
```

### Solution

In [ ]:
a = tf.reshape(tf.range(6), (2, 3))
print(a, a.dtype)
a = tf.cast(a, tf.float32)
b = tf.ones(3)

# b est diffusé (broadcasting) sur chaque ligne de a
print(a + b, (a + b).shape)
# Produit matriciel : (2, 3) @ (3, 2) -> (2, 2)
print(a @ tf.ones((3, 2)))

try:
  a + tf.ones(2)
except tf.errors.InvalidArgumentError as erreur:
  print("a + tf.ones(2) :", erreur)
try:
  a @ b
except tf.errors.InvalidArgumentError as erreur:
  print("a @ b :", erreur)

print(a[:, 1])  # deuxième colonne
print(a[-1])  # dernière ligne
print(tf.expand_dims(a, axis=0).shape)  # dimension de batch de taille 1

print(type(a.numpy()), tf.convert_to_tensor(a.numpy()).dtype)
tenseur_64 = tf.convert_to_tensor(np.ones(3))
print(tenseur_64.dtype)
try:
  a + tenseur_64
except tf.errors.InvalidArgumentError as erreur:
  print("a + tenseur_64 :", erreur)
print(tf.config.list_physical_devices("GPU"))

- `tf.range` crée des entiers 32 bits (`int32`) ; les réseaux de neurones calculent en flottants, par défaut `float32`.
- `a + b` a la forme (2, 3) et `a @ tf.ones((3, 2))` la forme (2, 2). Pour le broadcasting, les dimensions sont comparées en partant de la dernière : elles doivent être égales, ou l'une d'elles doit valoir 1 ou être absente. (2, 3) et (3,) sont compatibles, mais pas (2, 3) et (2,) : il faudrait la forme (2, 1), `tf.ones((2, 1))`.
- `a @ b` échoue aussi : en TensorFlow, `@` multiplie des matrices (au moins deux dimensions), alors que NumPy accepte un vecteur. Il faut `b` de forme (3, 1), ou `tf.linalg.matvec(a, b)`.
- `tf.expand_dims(a, axis=0)` a la forme (1, 2, 3). Les modèles traitent des batchs, la première dimension étant celle des exemples : pour prédire sur un seul exemple, on lui ajoute une dimension de batch de taille 1.
- `tf.convert_to_tensor` conserve le type NumPy : `float32` pour `a.numpy()`, mais `float64` pour `np.ones(3)`, le type par défaut de NumPy. TensorFlow ne convertit jamais un type en un autre de lui-même : `float32 + float64` lève une erreur, là où NumPy promeut en `float64`. D'où le `tf.cast`, ou `.astype("float32")` côté NumPy, que fera la préparation des images de MNIST.
- Sans GPU, la liste est vide et TensorFlow calcule sur le processeur ; avec un GPU, il l'utilise d'office.

## Régression linéaire avec `GradientTape`

Avant d'écrire un réseau de neurones, entraînons le plus simple des modèles, une droite $\hat{y} = wx + b$, par une descente de gradient écrite à la main. Les réseaux de ce TP suivront la même boucle, avec davantage de variables.

La cellule suivante crée les données : 100 points autour de la droite $y = 3x + 2$, avec un bruit gaussien d'écart type 1. Le but est de retrouver $w = 3$ et $b = 2$ à partir des seuls points.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 5, size=100).astype("float32")
y = (3 * x + 2 + rng.normal(0, 1, size=100)).astype("float32")

plt.scatter(x, y, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

*Créez deux `tf.Variable`, `w` et `b`, nulles au départ. À chacun de 500 pas, calculez sous un `tf.GradientTape` la perte, l'erreur quadratique moyenne (MSE) $\frac{1}{n} \sum_i (w x_i + b - y_i)^2$, puis ses gradients par rapport à `w` et `b` (`tape.gradient`), et retirez de chaque variable 0,05 fois son gradient (`assign_sub`). Gardez la perte de chaque pas dans une liste, tracez-la, et affichez `w`, `b` et la perte finale. Que se passe-t-il avec un learning rate de 0,1 ?*

**Documentation utile**

- [`tf.Variable`](https://www.tensorflow.org/api_docs/python/tf/Variable)
- [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape)
- [`tf.Variable.assign_sub`](https://www.tensorflow.org/api_docs/python/tf/Variable#assign_sub)
- [`tf.reduce_mean`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_mean)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tf.Variable(0.0)` : une variable scalaire, en `float32` comme `x` et `y`
- `tf.reduce_mean((w * x + b - y) ** 2)` : la MSE, calculée sur les 100 points à la fois, dans le bloc `with tf.GradientTape() as tape:` pour que la tape enregistre les opérations
- `tape.gradient(perte, [w, b])` : les deux gradients, dans l'ordre des variables
- `w.assign_sub(taux_apprentissage * dw)` : $w \leftarrow w - \eta \, \partial L / \partial w$, après le bloc de la tape
- `float(perte)` : la perte, en nombre Python, à ajouter à la liste
- `plt.yscale("log")` : une échelle logarithmique, qui montre à la fois les premiers pas et les derniers

Attendu : une perte d'environ 124 au premier pas, et de moins de 1 à la fin ; `w` et `b` proches de 3 et 2.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
w = tf.Variable(...)
b = tf.Variable(...)
taux_apprentissage = 0.05
pertes = []

for etape in range(500):
  with tf.GradientTape() as tape:
    perte = ...
  dw, db = tape.gradient(..., [w, b])
  w.assign_sub(...)
  b.assign_sub(...)
  pertes.append(float(perte))

plt.plot(pertes)
plt.yscale("log")
plt.xlabel("Pas")
plt.ylabel("MSE")
plt.show()
print(f"w = {w.numpy():.2f}, b = {b.numpy():.2f}, MSE = {perte:.2f}")
```

### Solution

In [ ]:
w = tf.Variable(0.0)
b = tf.Variable(0.0)
taux_apprentissage = 0.05
pertes = []

for etape in range(500):
  # Les opérations sur w et b, enregistrées par la tape
  with tf.GradientTape() as tape:
    perte = tf.reduce_mean((w * x + b - y) ** 2)  # MSE
  dw, db = tape.gradient(perte, [w, b])
  w.assign_sub(taux_apprentissage * dw)  # w ← w - η ∂L/∂w
  b.assign_sub(taux_apprentissage * db)
  pertes.append(float(perte))

plt.plot(pertes)
plt.yscale("log")
plt.xlabel("Pas")
plt.ylabel("MSE")
plt.title("Perte pendant la descente de gradient")
plt.show()
print(f"w = {w.numpy():.2f}, b = {b.numpy():.2f}, MSE = {perte:.2f}")

- La perte passe de 124 au premier pas à 1,6 au deuxième, puis se stabilise à 0,94 en une centaine de pas : $w = 2{,}97$ et $b = 1{,}99$. Ce sont, au centième près, les coefficients des moindres carrés de ces 100 points (`np.polyfit(x, y, 1)`) : la descente de gradient a trouvé la meilleure droite, et l'écart à 3 et 2 vient du bruit, pas de l'optimisation. La MSE finale, proche de 1, est la variance du bruit, que la droite ne peut pas expliquer.
- Avec un learning rate de 0,1, la perte grandit à chaque pas, jusqu'à l'infini (`inf`) : chaque pas dépasse le minimum et s'en éloigne davantage de l'autre côté. Le learning rate maximal dépend des données : ici, la descente diverge au-delà d'environ 0,094.
- Les réseaux de neurones s'entraînent de la même façon, avec `modele.trainable_variables` à la place de `[w, b]`, et des mini-batchs à la place de l'ensemble entier.

## Les données

Le TP Keras a exploré MNIST : 60 000 images d'entraînement et 10 000 de test, chacune un tableau `uint8` de 28 $\times$ 28 pixels, de 0 (fond) à 255 (encre), des labels de 0 à 9, et des classes à peu près équilibrées. La cellule suivante charge les données, affiche leurs formes et quelques exemples.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print(X_train.shape, X_train.dtype, y_train.shape, X_test.shape)

_, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(str(y_train[i]))
  ax.axis("off")
plt.show()

## Préparation des images

Trois transformations préparent les images pour le réseau :

1. Les convertir en `float32` : les variables du modèle sont en `float32`, et TensorFlow refuse de mélanger deux types.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt qu'entre $0$ et $255$.
3. Les mettre à plat : un réseau sans couche cachée prend un vecteur par exemple, de forme `(_, 28²)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples.

La fonction `transforme_images` les applique à un tableau ou à un tenseur d'images. Les images restent stockées brutes, en `uint8`, quatre fois plus légères qu'en `float32` : le pipeline `tf.data` les préparera batch par batch, et on préparera d'un coup un ensemble entier pour y évaluer un modèle.

In [ ]:
def transforme_images(images: np.ndarray | tf.Tensor) -> tf.Tensor:
  # (n, 28, 28) uint8, de 0 à 255 -> (n, 784) float32, de 0 à 1
  images = tf.cast(images, tf.float32) / 255
  return tf.reshape(images, (-1, 28 * 28))


print(transforme_images(X_train[:5]).shape, transforme_images(X_train[:5]).dtype)

## Séparation entraînement / validation

Pour choisir les hyper-paramètres (learning rate, taille du réseau…), on compare des modèles sur un ensemble de **validation**. L'ensemble de test est réservé à l'évaluation finale : s'en servir pour faire des choix rendrait cette évaluation trop optimiste.

On met de côté les 10 000 derniers exemples d'entraînement pour la validation :

In [ ]:
X_train, X_val = X_train[:50_000], X_train[50_000:]
y_train, y_val = y_train[:50_000], y_train[50_000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Réseau de neurones sans couche cachée

Un réseau de neurones sans couche cachée calcule $\text{softmax}(XW + b)$ : c'est la **régression logistique multinomiale**. `W` est la matrice des poids, multipliée à l'entrée, et `b` le vecteur des biais, ajouté ensuite. La softmax transforme les 10 scores de chaque exemple, ses *logits*, en probabilités.

### Initialisation des poids

*Complétez la fonction `initialise_poids(nb_ent, nb_sor)`, qui renvoie une `tf.Variable` de forme `(nb_ent, nb_sor)` tirée d'une loi normale (`tf.random.normal`), divisée par $\sqrt{n_\text{ent}}$.*

Cette division, l'initialisation de LeCun, donne aux poids une variance de $1 / n_\text{ent}$ : la sortie de la couche, une somme de $n_\text{ent}$ termes, reste de l'ordre de grandeur de ses entrées, quel que soit leur nombre.

**Documentation utile**

- [`tf.random.normal`](https://www.tensorflow.org/api_docs/python/tf/random/normal)

In [ ]:
def initialise_poids(nb_ent: int, nb_sor: int) -> tf.Variable:
  pass  # Votre code ici

#### Conseils supplémentaires

- `tf.random.normal((nb_ent, nb_sor))` : un tenseur de forme `(nb_ent, nb_sor)`, de moyenne 0 et d'écart type 1
- `nb_ent**0.5` : la racine carrée de `nb_ent`
- `tf.Variable(tenseur)` : une variable, que TensorFlow pourra mettre à jour

Attendu : `initialise_poids(784, 10)` a la forme (784, 10) et un écart type proche de $1 / 28 \approx 0{,}036$ (`tf.math.reduce_std`).

```python
def initialise_poids(nb_ent: int, nb_sor: int) -> tf.Variable:
  return tf.Variable(tf.random.normal(...) / ...)
```

#### Solution

In [ ]:
def initialise_poids(nb_ent: int, nb_sor: int) -> tf.Variable:
  # Variance 1 / nb_ent : des sorties de l'ordre de 1, quel que soit nb_ent
  return tf.Variable(tf.random.normal((nb_ent, nb_sor)) / nb_ent**0.5)


W = initialise_poids(784, 10)
print(W.shape, float(tf.math.reduce_std(W)))

### Définition du modèle

*Complétez la classe `ReseauSimple` : dans `__init__`, créez `self.W` avec `initialise_poids` et `self.b`, une `tf.Variable` de zéros de forme `(nb_sorties,)` ; `__call__` renvoie les logits $XW + b$.*

Le modèle ne calcule pas la softmax : la fonction de perte l'appliquera elle-même (`from_logits=True`), de façon plus stable numériquement.

In [ ]:
class ReseauSimple(tf.Module):
  def __init__(self, nb_entrees: int = 28**2, nb_sorties: int = 10) -> None:
    super().__init__()
    # Votre code ici

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    pass  # Votre code ici

#### Conseils supplémentaires

- `tf.Variable(tf.zeros(nb_sorties))` : les biais, nuls au départ
- `X @ self.W + self.b` : le produit matriciel, puis l'ajout des biais, diffusé sur chaque ligne

Attendu : `ReseauSimple()(transforme_images(X_train[:5]))` a la forme (5, 10), et `ReseauSimple().trainable_variables` contient 2 variables.

```python
class ReseauSimple(tf.Module):
  def __init__(self, nb_entrees: int = 28**2, nb_sorties: int = 10) -> None:
    super().__init__()
    self.W = initialise_poids(..., ...)
    self.b = tf.Variable(...)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    return ...
```

#### Solution

In [ ]:
class ReseauSimple(tf.Module):
  def __init__(self, nb_entrees: int = 28**2, nb_sorties: int = 10) -> None:
    super().__init__()
    self.W = initialise_poids(nb_entrees, nb_sorties)
    self.b = tf.Variable(tf.zeros(nb_sorties))

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # Les logits : la softmax est appliquée par la fonction de perte
    return X @ self.W + self.b


reseau_simple = ReseauSimple()
print(
  reseau_simple(transforme_images(X_train[:5])).shape,
  len(reseau_simple.trainable_variables),
)

### Calcul de la fonction de perte

La fonction de perte standard en classification multi-classes est l'entropie croisée. Keras la fournit pour des labels entiers (0 à 9, plutôt que des vecteurs one-hot), dans sa version dite *sparse* : `keras.losses.sparse_categorical_crossentropy`, avec `from_logits=True` puisque le modèle renvoie des logits. Elle rend une valeur par exemple : pour pouvoir l'interpréter directement, on en prend la moyenne avec `tf.reduce_mean`.

*Complétez la fonction `entropie_croisee_sparse`. Quelle valeur prend-elle pour un modèle qui n'a pas encore appris ?*

**Documentation utile**

- [`keras.losses.sparse_categorical_crossentropy`](https://keras.io/api/losses/probabilistic_losses/#sparse_categorical_crossentropy-function)

In [ ]:
def entropie_croisee_sparse(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  pass  # Votre code ici

#### Conseils supplémentaires

- `keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)` : les labels d'abord, puis les logits ; une valeur par exemple
- `tf.reduce_mean(...)` : leur moyenne, un scalaire

Un modèle qui n'a pas appris donne à peu près la même probabilité, 1/10, à chaque classe : sa perte vaut environ $-\log(1/10) = \log 10 \approx 2{,}3$.

```python
def entropie_croisee_sparse(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  entropies = keras.losses.sparse_categorical_crossentropy(..., ..., from_logits=...)
  return tf.reduce_mean(...)


print(entropie_croisee_sparse(y_train, ReseauSimple()(transforme_images(X_train))))
```

#### Solution

In [ ]:
def entropie_croisee_sparse(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  # from_logits=True : la softmax est calculée dans la perte, de façon stable
  entropies = keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)
  return tf.reduce_mean(entropies)


# Environ log(10) = 2,3 : le modèle répond presque au hasard
print(entropie_croisee_sparse(y_train, ReseauSimple()(transforme_images(X_train))))

### Métriques

Pour savoir si un modèle apprend, on mesure ses performances. Nous utiliserons la plus simple, mais aussi une des plus informatives : l'exactitude (*accuracy*), le nombre de bonnes prédictions sur le nombre total de prédictions.

*Complétez la fonction `exactitude`. La classe prédite est celle du plus grand logit (`tf.argmax`) : comparez-la aux labels, puis calculez la proportion de bonnes réponses.*

In [ ]:
def exactitude(y: np.ndarray, logits: tf.Tensor) -> float:
  pass  # Votre code ici

#### Conseils supplémentaires

- `tf.argmax(logits, axis=1)` : la classe prédite pour chaque exemple (la softmax ne change pas l'ordre des scores, inutile de la calculer)
- `predictions == y` : un tenseur de booléens
- `tf.cast(..., tf.float32)` puis `tf.reduce_mean` : la proportion de `True`
- `float(...)` : un nombre Python, plus simple à afficher et à stocker

Attendu : environ 0,1 pour un modèle qui n'a pas appris, une chance sur dix.

```python
def exactitude(y: np.ndarray, logits: tf.Tensor) -> float:
  predictions = tf.argmax(..., axis=...)
  return float(tf.reduce_mean(tf.cast(..., tf.float32)))


print(exactitude(y_val, ReseauSimple()(transforme_images(X_val))))
```

#### Solution

In [ ]:
def exactitude(y: np.ndarray, logits: tf.Tensor) -> float:
  predictions = tf.argmax(logits, axis=1)
  return float(tf.reduce_mean(tf.cast(predictions == y, tf.float32)))


print(exactitude(y_val, ReseauSimple()(transforme_images(X_val))))

## Création d'un dataset TensorFlow

Le modèle apprendra par mini-batchs : à chaque pas, un petit groupe d'exemples tirés au hasard. `tf.data` les prépare, à partir des images brutes.

*Complétez la fonction `fabrique_dataset`, qui crée un `tf.data.Dataset` à partir des tableaux `X` et `y` (`from_tensor_slices`), le mélange à chaque epoch (`shuffle`), le découpe en batchs `(X_batch, y_batch)` de `taille_lot` exemples (`batch`), prépare les images de chaque batch avec `transforme_images` (`map`), et prépare le batch suivant pendant le calcul (`prefetch`). Pourquoi appliquer `map` après `batch` plutôt qu'avant ?*

**Documentation utile**

- [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset)

In [ ]:
def fabrique_dataset(
  X: np.ndarray, y: np.ndarray, taille_lot: int = 128
) -> tf.data.Dataset:
  pass  # Votre code ici

### Conseils supplémentaires

- `tf.data.Dataset.from_tensor_slices((X, y))` : un dataset d'exemples `(x, y)`
- `.shuffle(len(X))` **avant** `.batch(taille_lot)` : on mélange les exemples ; après, on ne mélangerait que l'ordre des mêmes batchs. Le tampon de `len(X)` exemples donne un mélange complet, refait à chaque epoch.
- `.map(lambda X_batch, y_batch: (transforme_images(X_batch), y_batch), num_parallel_calls=tf.data.AUTOTUNE)` : la fonction reçoit les deux éléments de chaque batch, et ne transforme que les images
- `.prefetch(tf.data.AUTOTUNE)` : en dernier

Attendu : `fabrique_dataset(X_train, y_train)` compte 391 batchs (`len`), et le premier, `next(iter(...))`, a des images de forme (128, 784), en `float32`.

```python
def fabrique_dataset(
  X: np.ndarray, y: np.ndarray, taille_lot: int = 128
) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices(...)
  dataset = dataset.shuffle(...)
  dataset = dataset.batch(...)
  dataset = dataset.map(
    lambda X_batch, y_batch: (..., y_batch), num_parallel_calls=tf.data.AUTOTUNE
  )
  return dataset.prefetch(...)


dataset = fabrique_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, X_batch.dtype, y_batch.shape)
```

### Solution

In [ ]:
def fabrique_dataset(
  X: np.ndarray, y: np.ndarray, taille_lot: int = 128
) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices((X, y))
  # Mélange des exemples, avant le découpage en batchs, refait à chaque epoch
  dataset = dataset.shuffle(len(X))
  dataset = dataset.batch(taille_lot)
  # Préparation des images, un batch entier à la fois
  dataset = dataset.map(
    lambda X_batch, y_batch: (transforme_images(X_batch), y_batch),
    num_parallel_calls=tf.data.AUTOTUNE,
  )
  # Préparation du batch suivant pendant le calcul sur le batch courant
  return dataset.prefetch(tf.data.AUTOTUNE)


dataset = fabrique_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, X_batch.dtype, y_batch.shape)

Après `batch`, `map` appelle `transforme_images` une fois par batch, sur 128 images à la fois, plutôt qu'une fois par image : 391 appels par epoch au lieu de 50 000, et des opérations vectorisées. `transforme_images` accepte un batch, de forme (128, 28, 28), comme un ensemble entier. Le mélange, lui, porte sur les images brutes : son tampon de 50 000 images, en `uint8`, prend quatre fois moins de mémoire qu'en `float32`.

## Apprentissage

Voici une boucle d'apprentissage presque complète : elle parcourt les batchs à chaque epoch, puis mesure la perte et l'exactitude sur les ensembles d'entraînement et de validation, et trace leurs courbes à la fin.

*Implémentez la partie manquante : la mise à jour des variables du modèle pour un batch, par descente de gradient. Calculez la perte sous un `tf.GradientTape`, en lui ajoutant `regularisation(modele)` quand une régularisation est donnée (elle servira plus loin), puis ses gradients, et retirez de chaque variable `taux_apprentissage` fois son gradient avec `assign_sub`.*

**Documentation utile**

- [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape)
- [`tf.Variable.assign_sub`](https://www.tensorflow.org/api_docs/python/tf/Variable#assign_sub)

In [ ]:
FonctionPerte = Callable[[np.ndarray, tf.Tensor], tf.Tensor]
FonctionRegularisation = Callable[[tf.Module], tf.Tensor]


def entraine(
  modele: tf.Module,
  X_train: np.ndarray = X_train,
  y_train: np.ndarray = y_train,
  X_val: np.ndarray = X_val,
  y_val: np.ndarray = y_val,
  epoques: int = 10,
  taille_lot: int = 128,
  taux_apprentissage: float = 0.1,
  evaluer_tous_les: int = 1,
  perte: FonctionPerte = entropie_croisee_sparse,
  regularisation: FonctionRegularisation | None = None,
) -> tuple[list[float], ...]:
  # Les ensembles entiers, préparés une fois pour les évaluations
  X_train_prep, X_val_prep = transforme_images(X_train), transforme_images(X_val)
  y_train_pred = modele(X_train_prep)
  y_val_pred = modele(X_val_prep)
  exactitudes = [exactitude(y_train, y_train_pred)]
  exactitudes_val = [exactitude(y_val, y_val_pred)]
  pertes = [float(perte(y_train, y_train_pred))]
  pertes_val = [float(perte(y_val, y_val_pred))]

  print(
    f"Métriques initiales : exactitude {exactitudes[-1]:.4f}, "
    f"exactitude_val {exactitudes_val[-1]:.4f}, "
    f"perte {pertes[-1]:.4f}, "
    f"perte_val {pertes_val[-1]:.4f}"
  )

  dataset_ent = fabrique_dataset(X_train, y_train, taille_lot)

  for epoque in range(epoques):
    for X_batch, y_batch in dataset_ent:
      pass  # Votre code ici

    # Calcul et affichage des métriques toutes les `evaluer_tous_les` epochs
    if (epoque + 1) % evaluer_tous_les == 0:
      y_train_pred = modele(X_train_prep)
      y_val_pred = modele(X_val_prep)
      exactitudes.append(exactitude(y_train, y_train_pred))
      pertes.append(float(perte(y_train, y_train_pred)))
      exactitudes_val.append(exactitude(y_val, y_val_pred))
      pertes_val.append(float(perte(y_val, y_val_pred)))
      print(
        f"Epoch {epoque + 1} : exactitude {exactitudes[-1]:.4f}, "
        f"exactitude_val {exactitudes_val[-1]:.4f}, "
        f"perte {pertes[-1]:.4f}, "
        f"perte_val {pertes_val[-1]:.4f}"
      )

  graduations = np.arange(0, epoques + 1, evaluer_tous_les)

  plt.plot(graduations, pertes, label="Entraînement")
  plt.plot(graduations, pertes_val, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(graduations, exactitudes, label="Entraînement")
  plt.plot(graduations, exactitudes_val, label="Validation")
  plt.title("Exactitude pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return exactitudes, exactitudes_val, pertes, pertes_val

### Conseils supplémentaires

- `with tf.GradientTape() as tape:` : les opérations du bloc sont enregistrées
- `perte(y_batch, modele(X_batch))` : la perte du batch ; `+= regularisation(modele)` si `regularisation is not None`
- `tape.gradient(valeur_perte, modele.trainable_variables)` : la liste des gradients, dans l'ordre des variables
- `variable.assign_sub(taux_apprentissage * gradient)` : la mise à jour, pour chaque couple de `zip(gradients, modele.trainable_variables)`

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_batch, y_batch in dataset_ent:
  with tf.GradientTape() as tape:
    valeur_perte = ...
    if regularisation is not None:
      valeur_perte += ...
  gradients = tape.gradient(..., modele.trainable_variables)
  for gradient, variable in zip(gradients, modele.trainable_variables):
    variable.assign_sub(...)
```

### Solution

In [ ]:
FonctionPerte = Callable[[np.ndarray, tf.Tensor], tf.Tensor]
FonctionRegularisation = Callable[[tf.Module], tf.Tensor]


def entraine(
  modele: tf.Module,
  X_train: np.ndarray = X_train,
  y_train: np.ndarray = y_train,
  X_val: np.ndarray = X_val,
  y_val: np.ndarray = y_val,
  epoques: int = 10,
  taille_lot: int = 128,
  taux_apprentissage: float = 0.1,
  evaluer_tous_les: int = 1,
  perte: FonctionPerte = entropie_croisee_sparse,
  regularisation: FonctionRegularisation | None = None,
) -> tuple[list[float], ...]:
  # Les ensembles entiers, préparés une fois pour les évaluations
  X_train_prep, X_val_prep = transforme_images(X_train), transforme_images(X_val)
  y_train_pred = modele(X_train_prep)
  y_val_pred = modele(X_val_prep)
  exactitudes = [exactitude(y_train, y_train_pred)]
  exactitudes_val = [exactitude(y_val, y_val_pred)]
  pertes = [float(perte(y_train, y_train_pred))]
  pertes_val = [float(perte(y_val, y_val_pred))]

  print(
    f"Métriques initiales : exactitude {exactitudes[-1]:.4f}, "
    f"exactitude_val {exactitudes_val[-1]:.4f}, "
    f"perte {pertes[-1]:.4f}, "
    f"perte_val {pertes_val[-1]:.4f}"
  )

  dataset_ent = fabrique_dataset(X_train, y_train, taille_lot)

  for epoque in range(epoques):
    for X_batch, y_batch in dataset_ent:
      # Calcul de la perte, opérations enregistrées par la tape
      with tf.GradientTape() as tape:
        valeur_perte = perte(y_batch, modele(X_batch))
        if regularisation is not None:
          valeur_perte += regularisation(modele)

      # Gradient de la perte par rapport à chaque variable du modèle
      gradients = tape.gradient(valeur_perte, modele.trainable_variables)

      # Mise à jour de chaque variable selon la règle :
      # nouvelle_valeur = ancienne_valeur - taux_apprentissage * gradient
      for gradient, variable in zip(gradients, modele.trainable_variables):
        variable.assign_sub(taux_apprentissage * gradient)

    # Calcul et affichage des métriques toutes les `evaluer_tous_les` epochs
    if (epoque + 1) % evaluer_tous_les == 0:
      y_train_pred = modele(X_train_prep)
      y_val_pred = modele(X_val_prep)
      exactitudes.append(exactitude(y_train, y_train_pred))
      pertes.append(float(perte(y_train, y_train_pred)))
      exactitudes_val.append(exactitude(y_val, y_val_pred))
      pertes_val.append(float(perte(y_val, y_val_pred)))
      print(
        f"Epoch {epoque + 1} : exactitude {exactitudes[-1]:.4f}, "
        f"exactitude_val {exactitudes_val[-1]:.4f}, "
        f"perte {pertes[-1]:.4f}, "
        f"perte_val {pertes_val[-1]:.4f}"
      )

  graduations = np.arange(0, epoques + 1, evaluer_tous_les)

  plt.plot(graduations, pertes, label="Entraînement")
  plt.plot(graduations, pertes_val, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(graduations, exactitudes, label="Entraînement")
  plt.plot(graduations, exactitudes_val, label="Validation")
  plt.title("Exactitude pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return exactitudes, exactitudes_val, pertes, pertes_val

### Entraînement du réseau sans couche cachée

10 epochs, avec un learning rate de 0,5 :

In [ ]:
reseau_simple = ReseauSimple()
_ = entraine(reseau_simple, epoques=10, taux_apprentissage=0.5)

*Quelle exactitude de validation atteint ce modèle ? Progresse-t-il encore après quelques epochs ?*

Votre réponse ici

#### Solution

Environ 92 % d'exactitude en validation (de 92,2 à 92,8 % selon l'epoch). L'essentiel est appris dès la première epoch (91,4 %) ; ensuite, le modèle ne gagne plus qu'un point en 9 epochs, et oscille un peu. Les exactitudes d'entraînement et de validation restent égales : le modèle ne sur-apprend pas, il est trop simple. Une régression logistique sépare les classes par des frontières linéaires, ce qui plafonne vers 92 % sur MNIST.

## Avec un optimiseur Keras

La mise à jour écrite à la main ci-dessus (`variable.assign_sub(taux_apprentissage * gradient)`) est exactement ce que fait l'optimiseur `keras.optimizers.SGD`. Les optimiseurs de Keras s'utilisent sans modèle Keras, sur de simples `tf.Variable` : `optimiseur.apply_gradients(zip(gradients, variables))` remplace la boucle de mise à jour.

*Complétez `etape_entrainement`, un pas d'entraînement de `modele` par `optimiseur` sur un batch : la perte du batch sous un `tf.GradientTape`, ses gradients, puis un pas de l'optimiseur avec `apply_gradients` ; la fonction renvoie la perte. `entraine_avec_optimiseur`, qui la suit, l'appelle pour chaque batch ; elle resservira plus loin. La cellule qui suit la solution entraîne un `ReseauSimple` avec `keras.optimizers.SGD` et le même learning rate que précédemment, puis un autre avec `keras.optimizers.Adam`.*

**Documentation utile**

- [`keras.optimizers.SGD`](https://keras.io/api/optimizers/sgd/)
- [`apply_gradients`](https://keras.io/api/optimizers/#applygradients-method)

In [ ]:
def etape_entrainement(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  X_batch: tf.Tensor,
  y_batch: tf.Tensor,
) -> tf.Tensor:
  pass  # Votre code ici


def entraine_avec_optimiseur(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  epoques: int = 5,
  taille_lot: int = 128,
) -> None:
  dataset_ent = fabrique_dataset(X_train, y_train, taille_lot)
  X_val_prep = transforme_images(X_val)
  for epoque in range(epoques):
    for X_batch, y_batch in dataset_ent:
      etape_entrainement(modele, optimiseur, X_batch, y_batch)

    exactitude_val = exactitude(y_val, modele(X_val_prep))
    print(f"Epoch {epoque + 1} : exactitude_val {exactitude_val:.4f}")

### Conseils supplémentaires

- La perte et les gradients se calculent comme dans `entraine`, sans régularisation, avec `entropie_croisee_sparse`
- `optimiseur.apply_gradients(zip(gradients, modele.trainable_variables))` : un pas de l'optimiseur sur toutes les variables

Un optimiseur Keras crée ses propres variables (les moyennes d'Adam, par exemple) pour les variables qu'il voit au premier pas : un nouveau modèle demande un nouvel optimiseur.

```python
def etape_entrainement(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  X_batch: tf.Tensor,
  y_batch: tf.Tensor,
) -> tf.Tensor:
  with tf.GradientTape() as tape:
    valeur_perte = entropie_croisee_sparse(..., ...)
  gradients = tape.gradient(..., ...)
  optimiseur.apply_gradients(zip(..., ...))
  return valeur_perte
```

### Solution

In [ ]:
def etape_entrainement(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  X_batch: tf.Tensor,
  y_batch: tf.Tensor,
) -> tf.Tensor:
  with tf.GradientTape() as tape:
    valeur_perte = entropie_croisee_sparse(y_batch, modele(X_batch))
  gradients = tape.gradient(valeur_perte, modele.trainable_variables)
  # Un pas de l'optimiseur, qui remplace les assign_sub écrits à la main
  optimiseur.apply_gradients(zip(gradients, modele.trainable_variables))
  return valeur_perte

### Comparaison de SGD et d'Adam

5 epochs avec chacun : SGD avec le learning rate de la boucle écrite à la main, Adam avec son learning rate usuel, bien plus petit.

In [ ]:
print("SGD")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(reseau_simple, keras.optimizers.SGD(learning_rate=0.5))

print("Adam")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(reseau_simple, keras.optimizers.Adam(learning_rate=1e-3))

*Obtenez-vous avec SGD les performances de la boucle écrite à la main ? Que change Adam ?*

Votre réponse ici

#### Solution

Avec SGD et le même learning rate, on retrouve les performances de la boucle écrite à la main (environ 92 % en 5 epochs) : `apply_gradients` applique la même règle de mise à jour. Changer d'algorithme d'optimisation ne demande plus que de changer d'optimiseur. Adam adapte le pas de chaque variable à l'historique de ses gradients : avec un learning rate 500 fois plus petit, il arrive au même résultat. Sur ce modèle simple, il n'apporte rien de plus ; son intérêt est que la valeur 10⁻³ convient à la plupart des réseaux, quand celle de SGD est à chercher pour chacun.

## Compiler le pas d'entraînement avec `tf.function`

Jusqu'ici, TensorFlow exécute chaque opération dès que Python l'appelle (mode *eager*). Décorée par `tf.function`, une fonction est d'abord *tracée* : son code Python s'exécute une fois, et TensorFlow enregistre ses opérations dans un graphe, qu'il exécute ensuite sans repasser par Python. Un appel dont les arguments ont une autre *signature* (forme ou `dtype` d'un tenseur, valeur d'un argument Python) déclenche un nouveau traçage.

La cellule suivante définit `etape`, un pas d'entraînement par SGD d'un nouveau `ReseauSimple`, qui compte combien de fois son code Python s'exécute, et `chronometre_epoque(fonction_etape, nom)`, qui chronomètre une epoch faite avec `fonction_etape` et affiche ce compte. Elle chronomètre une epoch en mode eager.

1. *Créez `etape_compilee`, la version compilée de `etape` par `tf.function`, et chronométrez deux epochs avec elle. Laquelle des deux versions est la plus rapide, et pourquoi ? Combien de fois le code Python de `etape` s'exécute-t-il à chaque epoch, et pourquoi ?*
2. *Créez `etape_signature`, compilée avec un `input_signature` qui accepte des batchs de toute taille, et chronométrez une epoch avec elle. Combien de traçages reste-t-il ?*

**Documentation utile**

- [`tf.function`](https://www.tensorflow.org/api_docs/python/tf/function)
- [`tf.TensorSpec`](https://www.tensorflow.org/api_docs/python/tf/TensorSpec)

In [ ]:
modele = ReseauSimple()
optimiseur = keras.optimizers.SGD(learning_rate=0.5)
dataset_ent = fabrique_dataset(X_train, y_train)
nb_executions = 0


def etape(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
  global nb_executions
  # Code Python : exécuté à chaque appel en mode eager, aux traçages seulement
  # une fois compilé
  nb_executions += 1
  return etape_entrainement(modele, optimiseur, X_batch, y_batch)


def chronometre_epoque(
  fonction_etape: Callable[[tf.Tensor, tf.Tensor], tf.Tensor], nom: str
) -> None:
  global nb_executions
  nb_executions = 0
  debut = time.perf_counter()
  for X_batch, y_batch in dataset_ent:
    fonction_etape(X_batch, y_batch)
  duree = time.perf_counter() - debut
  print(f"{nom} : {duree:.2f} s, code Python de etape exécuté {nb_executions} fois")


chronometre_epoque(etape, "Eager")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tf.function(etape)` : la version compilée, à appeler comme `etape`
- `chronometre_epoque(etape_compilee, "Compilé, epoch 1")` : la durée d'une epoch avec elle, et le nombre d'exécutions du code Python
- `tf.TensorSpec(shape=(None, 784), dtype=tf.float32)` : un tenseur de forme `(n, 784)`, quel que soit `n`, comme les images d'un batch ; les labels d'un batch sont des `tf.uint8` de forme `(n,)`
- `tf.function(etape, input_signature=[...])` : une spécification par argument, dans l'ordre

```python
etape_compilee = tf.function(...)
chronometre_epoque(etape_compilee, "Compilé, epoch 1")
chronometre_epoque(..., "Compilé, epoch 2")

etape_signature = tf.function(
  etape,
  input_signature=[
    tf.TensorSpec(shape=(None, ...), dtype=...),
    tf.TensorSpec(shape=..., dtype=...),
  ],
)
chronometre_epoque(..., "Avec input_signature")
```

### Solution

In [ ]:
etape_compilee = tf.function(etape)
# Le premier appel de chaque forme de batch trace la fonction
chronometre_epoque(etape_compilee, "Compilé, epoch 1")
chronometre_epoque(etape_compilee, "Compilé, epoch 2")

# Une seule signature, pour des batchs de toute taille
etape_signature = tf.function(
  etape,
  input_signature=[
    tf.TensorSpec(shape=(None, 28 * 28), dtype=tf.float32),
    tf.TensorSpec(shape=(None,), dtype=tf.uint8),
  ],
)
chronometre_epoque(etape_signature, "Avec input_signature")

La version compilée est 2 à 4 fois plus rapide, selon la machine et sa charge : le graphe s'exécute sans repasser par Python à chaque opération, et TensorFlow peut l'optimiser (fusion d'opérations). Le gain est d'autant plus grand que les opérations sont petites : ici, le surcoût de Python pèse lourd face à des produits matriciels de 128 exemples.

En mode eager, le code Python de `etape` s'exécute à chaque pas : 391 fois par epoch. Compilée, `etape` ne l'exécute qu'à ses traçages : 2 fois à la première epoch, parce que le dernier batch, de 80 exemples (50 000 = 390 $\times$ 128 + 80), a une autre forme que les autres et déclenche un second traçage ; 0 fois à la seconde, les deux graphes étant en cache. Un `print` dans une fonction compilée fait de même : il n'affiche plus rien après les traçages, quand `tf.print` s'exécute à chaque appel.

Avec `input_signature`, la première dimension est libre (`None`) : un seul graphe sert à toutes les tailles de batch, et `etape_signature` n'est tracée qu'une fois. Un traçage ne coûte qu'une fraction de seconde ; en revanche, une fonction appelée avec des formes toujours nouvelles serait retracée à chaque appel, et plus lente qu'en mode eager. `drop_remainder=True` dans `batch` éviterait aussi le second traçage, au prix des 80 derniers exemples de chaque epoch.

## Passage à des réseaux profonds

Pour passer à un perceptron multicouche, on crée une matrice de poids et un vecteur de biais par couche. La fonction suivante les range dans une liste, un dictionnaire `{"W": ..., "b": ...}` par couche :

In [ ]:
def cree_poids_pmc(
  nb_entrees: int, nb_sorties: int, tailles_cachees: tuple[int, ...] = (512,)
) -> list[dict[str, tf.Variable]]:
  couches = []
  # La taille d'entrée d'une couche est la taille de sortie de la précédente
  taille_precedente = nb_entrees
  for taille in [*tailles_cachees, nb_sorties]:
    couches.append(
      {
        "W": initialise_poids(taille_precedente, taille),
        "b": tf.Variable(tf.zeros(taille)),
      }
    )
    taille_precedente = taille
  return couches

La sortie du modèle se calcule maintenant couche par couche : chaque couche prend en entrée le résultat de la précédente.

*Complétez `PMC.__call__`. Chaque couche cachée calcule $\text{ReLU}(XW + b)$ avec `tf.nn.relu` ; la dernière renvoie les logits $XW + b$, sans activation. Les variables de la couche `i` sont `self.couches[i]["W"]` et `self.couches[i]["b"]`.*

In [ ]:
class PMC(tf.Module):
  def __init__(
    self,
    tailles_cachees: tuple[int, ...] = (512,),
    nb_entrees: int = 28**2,
    nb_sorties: int = 10,
  ) -> None:
    super().__init__()
    self.couches = cree_poids_pmc(nb_entrees, nb_sorties, tailles_cachees)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    pass  # Votre code ici

### Conseils supplémentaires

- `self.couches[:-1]` : les couches cachées ; `self.couches[-1]` : la couche de sortie
- `tf.nn.relu(X @ couche["W"] + couche["b"])` : une couche cachée

Attendu : `PMC()(transforme_images(X_train[:5]))` a la forme (5, 10), et `PMC((256, 128))` a 6 variables (`trainable_variables`).

```python
class PMC(tf.Module):
  def __init__(
    self,
    tailles_cachees: tuple[int, ...] = (512,),
    nb_entrees: int = 28**2,
    nb_sorties: int = 10,
  ) -> None:
    super().__init__()
    self.couches = cree_poids_pmc(nb_entrees, nb_sorties, tailles_cachees)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    for couche in self.couches[:-1]:
      X = tf.nn.relu(...)
    couche_sortie = self.couches[-1]
    return ...
```

### Solution

In [ ]:
class PMC(tf.Module):
  def __init__(
    self,
    tailles_cachees: tuple[int, ...] = (512,),
    nb_entrees: int = 28**2,
    nb_sorties: int = 10,
  ) -> None:
    super().__init__()
    self.couches = cree_poids_pmc(nb_entrees, nb_sorties, tailles_cachees)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # Couches cachées : transformation affine, puis ReLU
    for couche in self.couches[:-1]:
      X = tf.nn.relu(X @ couche["W"] + couche["b"])
    # Couche de sortie : les logits, sans activation
    couche_sortie = self.couches[-1]
    return X @ couche_sortie["W"] + couche_sortie["b"]


print(
  PMC()(transforme_images(X_train[:5])).shape, len(PMC((256, 128)).trainable_variables)
)

## Entraînement du réseau profond

Une couche cachée de 512 neurones, 10 epochs, un learning rate de 0,5 :

In [ ]:
pmc = PMC()
_ = entraine(pmc, epoques=10, taux_apprentissage=0.5)

*Que constatez-vous en comparant ce réseau au réseau sans couche cachée ?*

Votre réponse ici

### Solution

Le réseau à une couche cachée fait nettement mieux que le réseau sans couche cachée : environ 98 % d'exactitude en validation, contre 92 %, soit une erreur divisée par 4. La couche cachée et sa ReLU rendent le modèle non linéaire. L'exactitude d'entraînement dépasse 99 % et s'écarte de celle de validation : le réseau commence à sur-apprendre, mais sa perte de validation baisse encore.

C'est en partie grâce à l'initialisation des poids, de variance $1 / n_\text{ent}$. Pour vous en convaincre, retirez la division par `nb_ent**0.5` dans `initialise_poids` et relancez : les logits initiaux ont un écart type de l'ordre de 100, la perte initiale vaut des centaines au lieu de 2,3, et le réseau reste loin derrière après 2 epochs.

## Régularisation

Le PMC précédent sur-apprend peu : sa perte de validation baisse encore à la dixième epoch, car 50 000 exemples suffisent à ses 400 000 poids. Le sur-apprentissage apparaît quand le réseau est grand par rapport aux données : la cellule qui suit la solution entraîne le même PMC sur les 1 000 premiers exemples seulement, 100 epochs, sans puis avec régularisation.

La régularisation L2 ajoute à la perte une pénalité, $\lambda \sum w^2$ : $\lambda$ fois la somme des carrés des poids, qui pousse les poids vers 0. On ne pénalise que les poids `W`, pas les biais : ils ne multiplient pas l'entrée, et les contraindre n'aide pas à généraliser.

*Complétez la fonction interne de `l2`, qui renvoie la pénalité d'un `PMC` : la somme des carrés des matrices `W` de toutes ses couches (`modele.couches`), multipliée par `coefficient_lambda`.*

**Documentation utile**

- [`tf.reduce_sum`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_sum)

In [ ]:
def l2(coefficient_lambda: float) -> FonctionRegularisation:
  def penalite(modele: PMC) -> tf.Tensor:
    pass  # Votre code ici

  return penalite

### Conseils supplémentaires

- `tf.reduce_sum(couche["W"] ** 2)` : la somme des carrés d'une matrice
- `sum(...)` de Python, ou `tf.add_n([...])`, pour additionner celles de toutes les couches

Attendu : pour un `PMC()` neuf, `l2(1.0)(PMC())` vaut environ 522 (512 pour la première couche, 10 pour la seconde : chacune contribue à peu près son nombre de colonnes, puisque chaque poids a une variance de $1 / n_\text{ent}$).

```python
def l2(coefficient_lambda: float) -> FonctionRegularisation:
  def penalite(modele: PMC) -> tf.Tensor:
    norme_poids = sum(tf.reduce_sum(...) for couche in modele.couches)
    return ... * norme_poids

  return penalite
```

### Solution

In [ ]:
def l2(coefficient_lambda: float) -> FonctionRegularisation:
  def penalite(modele: PMC) -> tf.Tensor:
    # Les poids W seulement, pas les biais
    norme_poids = sum(tf.reduce_sum(couche["W"] ** 2) for couche in modele.couches)
    return coefficient_lambda * norme_poids

  return penalite


print(float(l2(1.0)(PMC())))

### Sur-apprentissage sur 1 000 exemples

Le même PMC, avec le même learning rate de 0,5, sur les 1 000 premiers exemples d'entraînement, sans régularisation puis avec une pénalité L2 de $\lambda$ = 10⁻³ ; la validation reste faite sur les 10 000 exemples mis de côté.

In [ ]:
X_petit, y_petit = X_train[:1000], y_train[:1000]

print("Sans régularisation")
_ = entraine(
  PMC(),
  X_train=X_petit,
  y_train=y_petit,
  epoques=100,
  taux_apprentissage=0.5,
  evaluer_tous_les=10,
)

print("Avec une pénalité L2")
_ = entraine(
  PMC(),
  X_train=X_petit,
  y_train=y_petit,
  epoques=100,
  taux_apprentissage=0.5,
  evaluer_tous_les=10,
  regularisation=l2(1e-3),
)

*Comparez les deux entraînements : l'écart entre entraînement et validation, la perte de validation au fil des epochs, l'exactitude de validation finale.*

Votre réponse ici

#### Solution

Sans régularisation, le réseau apprend par cœur les 1 000 exemples : 100 % d'exactitude d'entraînement dès la vingtième epoch, une perte d'entraînement qui tend vers 0, et une exactitude de validation bloquée vers 89,5 %. La perte de validation passe par un minimum vers la vingtième epoch, puis remonte : le réseau devient de plus en plus sûr de ses réponses, y compris des fausses. C'est le sur-apprentissage.

Avec la pénalité L2, la perte d'entraînement reste au-dessus de 0,02 (les poids ne peuvent plus grandir librement) et la perte de validation ne remonte plus. L'exactitude de validation gagne peu, quelques dixièmes de point : sur si peu de données, la régularisation limite surtout l'excès de confiance du modèle. Rien ne remplace davantage de données : le même réseau atteint 98 % avec les 50 000 exemples.

## Suivre l'entraînement avec TensorBoard

Dans le TP Keras, le callback `keras.callbacks.TensorBoard` écrivait à chaque epoch les métriques de `fit` dans un journal, que TensorBoard affiche. Sans `fit`, on écrit ce journal soi-même :

- `tf.summary.create_file_writer(dossier)` crée un journal dans `dossier` ;
- dans un bloc `with journal.as_default():`, `tf.summary.scalar(nom, valeur, step=epoque)` ajoute un point à la courbe `nom`.

Un dossier par essai : TensorBoard superpose leurs courbes, ce qui permet de comparer des hyper-paramètres sur l'ensemble de validation, jamais sur le test.

*Complétez `entraine_avec_journal` : à chaque epoch, faites un pas d'entraînement par batch avec `etape_entrainement`, compilée par `tf.function`, puis écrivez dans le journal la perte moyenne de l'epoch (`perte`) et l'exactitude de validation (`exactitude_val`). La fonction renvoie la dernière exactitude de validation. La cellule qui suit la solution entraîne un PMC par SGD avec trois learning rates, chacun dans son dossier de `journaux/`, puis ouvre TensorBoard.*

**Documentation utile**

- [`tf.summary.create_file_writer`](https://www.tensorflow.org/api_docs/python/tf/summary/create_file_writer)
- [`tf.summary.scalar`](https://www.tensorflow.org/api_docs/python/tf/summary/scalar)

In [ ]:
def entraine_avec_journal(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  dossier: str,
  epoques: int = 5,
) -> float:
  pass  # Votre code ici

### Conseils supplémentaires

- Un pas compilé, propre à ce modèle et à cet optimiseur, défini dans la fonction :

```python
@tf.function
def etape(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
  return etape_entrainement(modele, optimiseur, X_batch, y_batch)
```

- `tf.reduce_mean(pertes)` : la perte moyenne de l'epoch, à partir de la liste des pertes de ses batchs
- `exactitude(y_val, modele(X_val_prep))`, avec `X_val_prep = transforme_images(X_val)` préparé une fois
- `step=epoque + 1` : l'abscisse du point, en epochs

```python
def entraine_avec_journal(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  dossier: str,
  epoques: int = 5,
) -> float:
  journal = tf.summary.create_file_writer(...)

  @tf.function
  def etape(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
    return etape_entrainement(modele, optimiseur, X_batch, y_batch)

  dataset_ent = fabrique_dataset(X_train, y_train)
  X_val_prep = transforme_images(X_val)
  for epoque in range(epoques):
    pertes = [etape(X_batch, y_batch) for X_batch, y_batch in dataset_ent]
    exactitude_val = exactitude(y_val, modele(X_val_prep))
    with journal.as_default():
      tf.summary.scalar("perte", ..., step=epoque + 1)
      tf.summary.scalar(..., ..., step=...)
  return exactitude_val
```

### Solution

In [ ]:
def entraine_avec_journal(
  modele: tf.Module,
  optimiseur: keras.optimizers.Optimizer,
  dossier: str,
  epoques: int = 5,
) -> float:
  journal = tf.summary.create_file_writer(dossier)

  # Un pas compilé propre à ce modèle et à cet optimiseur
  @tf.function
  def etape(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
    return etape_entrainement(modele, optimiseur, X_batch, y_batch)

  dataset_ent = fabrique_dataset(X_train, y_train)
  X_val_prep = transforme_images(X_val)
  for epoque in range(epoques):
    pertes = [etape(X_batch, y_batch) for X_batch, y_batch in dataset_ent]
    exactitude_val = exactitude(y_val, modele(X_val_prep))
    # Un point par courbe et par epoch
    with journal.as_default():
      tf.summary.scalar("perte", tf.reduce_mean(pertes), step=epoque + 1)
      tf.summary.scalar("exactitude_val", exactitude_val, step=epoque + 1)
    print(f"Epoch {epoque + 1} : exactitude_val {exactitude_val:.4f}")
  return exactitude_val

### Comparaison de trois learning rates

Un PMC par learning rate, 5 epochs chacun, journalisés dans `journaux/sgd_<learning rate>` :

In [ ]:
pmcs, exactitudes_val = {}, {}
for taux in (0.01, 0.1, 0.5):
  print(f"Learning rate {taux}")
  pmcs[taux] = PMC()
  optimiseur = keras.optimizers.SGD(learning_rate=taux)
  exactitudes_val[taux] = entraine_avec_journal(
    pmcs[taux], optimiseur, f"journaux/sgd_{taux}"
  )

In [ ]:
%load_ext tensorboard
%tensorboard --logdir journaux

*Dans TensorBoard, quel learning rate donne la meilleure exactitude de validation ? Que montrent les courbes de perte ?*

Votre réponse ici

#### Solution

0,5 : environ 97,7 % d'exactitude de validation en 5 epochs, contre 95,7 % pour 0,1 et 91 % pour 0,01 (courbe `exactitude_val`). Les courbes de `perte` montrent pourquoi : plus le learning rate est petit, plus la perte d'entraînement descend lentement. À 0,01, le réseau est loin d'avoir fini d'apprendre en 5 epochs : comparer des essais aussi courts favorise ce qui apprend vite, pas forcément ce qui finit le mieux.

TensorBoard relit les journaux à mesure qu'ils s'écrivent : on peut suivre un entraînement en cours. Relancer la cellule ajoute de nouveaux journaux aux anciens, dans les mêmes dossiers : supprimez `journaux/` pour repartir de zéro.

## Évaluation finale sur l'ensemble de test

L'ensemble de test n'a servi à rien jusqu'ici : ni à l'entraînement, ni au choix du learning rate. C'est ce qui rend son évaluation fiable.

*Prenez `pmc`, le PMC entraîné avec le learning rate de la meilleure exactitude de validation, et calculez son exactitude sur `X_test`. Est-elle proche de son exactitude de validation ?*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `max(exactitudes_val, key=exactitudes_val.get)` : la clé de la plus grande valeur du dictionnaire, ici le meilleur learning rate
- `pmcs[meilleur_taux]` : le PMC entraîné avec lui
- `exactitude(y_test, pmc(transforme_images(X_test)))` : l'exactitude de test

```python
meilleur_taux = max(..., key=...)
pmc = pmcs[...]
print(f"Exactitude sur l'ensemble de test : {exactitude(..., ...):.4f}")
```

### Solution

In [ ]:
# Le learning rate de la meilleure exactitude de validation, et son PMC
meilleur_taux = max(exactitudes_val, key=exactitudes_val.get)
pmc = pmcs[meilleur_taux]
print(f"Learning rate retenu : {meilleur_taux}")
print(f"Exactitude sur l'ensemble de validation : {exactitudes_val[meilleur_taux]:.4f}")
print(
  f"Exactitude sur l'ensemble de test : {exactitude(y_test, pmc(transforme_images(X_test))):.4f}"
)

L'exactitude de test est proche de l'exactitude de validation (ici 97,7 % pour l'une comme pour l'autre). La validation a servi à choisir le learning rate : son exactitude est en principe légèrement optimiste, d'autant plus qu'on a comparé plus d'essais. L'ensemble de test, qui n'a servi à aucun choix, donne une estimation fiable des performances du modèle sur de nouvelles données.

## Sauvegarder et restaurer le modèle

Un entraînement coûte cher : on sauvegarde le modèle entraîné pour le réutiliser sans le réentraîner. `tf.train.Checkpoint` enregistre les valeurs des variables d'un objet, retrouvées par leurs noms d'attributs.

*Sauvegardez `pmc` dans `points_controle/pmc`. Créez `restaure`, un nouveau `PMC` de même architecture, et affichez son exactitude de test avant et après y avoir restauré le point de contrôle.*

**Documentation utile**

- [`tf.train.Checkpoint`](https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tf.train.Checkpoint(modele=pmc).save("points_controle/pmc")` : la sauvegarde ; elle renvoie le chemin du point de contrôle, `points_controle/pmc-1` (le numéro augmente à chaque sauvegarde)
- `tf.train.Checkpoint(modele=restaure).restore(chemin)` : la restauration ; le nom d'argument, `modele`, est libre, mais doit être le même qu'à la sauvegarde
- `restaure` doit avoir la même architecture que `pmc` : un `PMC()`, avec ses tailles par défaut

```python
chemin = tf.train.Checkpoint(modele=...).save("points_controle/pmc")
restaure = PMC()
X_test_prep = transforme_images(X_test)
print(exactitude(y_test, restaure(X_test_prep)))
tf.train.Checkpoint(modele=...).restore(...)
print(exactitude(..., ...))
```

### Solution

In [ ]:
chemin = tf.train.Checkpoint(modele=pmc).save("points_controle/pmc")
print(chemin)

# Nouveaux poids aléatoires, même architecture que pmc
restaure = PMC()
X_test_prep = transforme_images(X_test)
print(f"Avant restauration : {exactitude(y_test, restaure(X_test_prep)):.4f}")
tf.train.Checkpoint(modele=restaure).restore(chemin)
print(f"Après restauration : {exactitude(y_test, restaure(X_test_prep)):.4f}")
print(f"Modèle sauvegardé : {exactitude(y_test, pmc(X_test_prep)):.4f}")

Avant restauration, `restaure` répond presque au hasard (de l'ordre de 10 %, ses poids sont aléatoires). Après, il retrouve à l'identique l'exactitude de `pmc` : le point de contrôle a rendu à chaque variable sa valeur, retrouvée par son chemin d'attributs (`modele`, puis `couches`, puis l'indice et la clé). D'où l'importance d'une architecture identique : un `PMC` d'une autre taille aurait des variables de formes différentes, et la restauration échouerait.

Un point de contrôle ne contient que des valeurs, pas le code du modèle : pour le restaurer, il faut la classe `PMC`. Pour déployer un modèle sans son code Python, TensorFlow a un autre format, le SavedModel (`tf.saved_model.save`), qui garde le graphe des méthodes compilées par `tf.function`.